<style>
    pre {
        white-space: pre-wrap;
        word-wrap: break-word;
    }
</style>

<div style="display:flex; justify-content:space-around; align-items:center; background-color:#cccccc; padding:5px; border:2px solid #333333;">
    <a href="https://www.um.es/web/estudios/grados/ciencia-ingenieria-datos/" target="_blank">
    <img src="https://www.um.es/documents/1073494/42130150/LogosimboloUMU-positivo.png" alt="UMU" style="height:200px; width:auto;">
    <a href="https://estudios.upct.es/grado/5251/inicio" target="_blank">
    <img src="https://www.upct.es/contenido/universidad/galeria/identidad-2021/logos/logos-upct/marca-upct/marca-principal/horizontal/azul.png" alt="UPCT" style="height:145px; width:auto;">
</div>

# Del Sinograma al Diagnóstico
## Análisis de Tumores Pulmonares en CT con MONAI

**Autores:** Francisco Javier Mercader Martínez, Chaimae Chahlal, Hiba Chakour

**Universidad Politécnica de Cartagena** — Curso 2025/2026

**Asignatura:** Procesamiento de imagen en Medicina (UMU-UPCT)

---

### Resumen

En este proyecto hemos montado un pipeline completo para analizar tumores pulmonares en imágenes CT, cubriendo desde la exploración básica de los datos hasta la segmentación automática con deep learning y el análisis radiómico. Utilizamos MONAI como framework principal y el dataset MSD Task06\_Lung (96 volúmenes CT con anotaciones de tumor). El trabajo conecta los cuatro bloques prácticos de la asignatura: exploración de formatos médicos y ventaneo CT (P01), simulación de reconstrucción tomográfica (P02), registro inter-paciente con SimpleITK (P03), y segmentación con MONAI, radiómica y experimentos de escasez de datos (P04).

## Motivación

El cáncer de pulmón es la primera causa de muerte por cáncer en el mundo. En el día a día de un hospital, cuando un paciente le hacen un CT de tórax y le encuentran un tumor, el radiólogo tiene que delimitar manualmente la región tumoral para que los oncólogos puedan planificar el tratamiento. Este proceso es tedioso, subjetivo (dos radiólogos pueden dibujar contornos diferentes) y consume mucho tiempo. Automatizarlo con inteligencia artificial puede ayudar bastante.

Nosotros pretendemos resolver este problema clínico (eso requeriría años de desarrollo y validación), pero sí queremos entender y practicar las ténicas que hay detrás.

### Correspondencia con las prácticas

| Módulo | Contenido del curso | Fase del proyecto |
|:------:|---------------------|-------------------|
| P01 | DICOM/NIfTI, HU, ventaneo | Fase 1: Exploración |
| P02 | Reconstrucción CT (Radon, FBP) | Fase 2: Reconstrucción |
| P03 | Registro rígido y afín | Fase 3: Registro |
| P04 | MONAI, DL, radiómica | Fases 4–7 |

In [ ]:
# ==============================================================================
# CONFIGURACIÓN Y DEPENDENCIAS
# ==============================================================================

import sys, importlib, subprocess
import warnings

print(f"Python version: {sys.version}\n")

# ── 1. Verificar / instalar dependencias ────────────────────────────────────


def ensure(module, pip_name=None, version_attr="__version__"):
    name = pip_name or module
    try:
        mod = importlib.import_module(module)
    except ImportError:
        print(f"[!] Instalando {name}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", name]
        )
        mod = importlib.import_module(module)
    ver = None if version_attr is None else getattr(mod, version_attr, None)
    print(f"[OK] {module}" + (f" {ver}" if ver else ""))


def ensure_radiomics():
    try:
        import radiomics
    except ImportError:
        print("[!] Instalando pyradiomics desde GitHub...")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "git+https://github.com/AIM-Harvard/pyradiomics.git",
            ]
        )
        import radiomics
    ver = getattr(radiomics, "__version__", None)
    print(f"[OK] radiomics" + (f" {ver}" if ver else ""))


print("Verificando librerías...")

warnings.filterwarnings("ignore")

# Base
ensure("numpy")
ensure("matplotlib", version_attr=None)
ensure("scipy")
ensure("PIL", "pillow", version_attr=None)
ensure("sklearn", "scikit-learn")
ensure("pandas")
ensure("nibabel")
ensure("cv2", "opencv-python", version_attr=None)
# Deep Learning / modelos
ensure("torch")
ensure("torchvision")
ensure("transformers")
ensure("huggingface_hub")
ensure("accelerate")
ensure("sentencepiece")
ensure("einops")
ensure("open_clip", "open_clip_torch", version_attr=None)
# Interfaces
ensure("gradio")
# IA Médica
ensure("monai")
ensure("SimpleITK")
ensure("fire", version_attr=None)
# Radiómica
ensure_radiomics()

import os
import json
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap, Normalize
from matplotlib.figure import Figure
from mpl_toolkits.axes_grid1 import make_axes_locatable

import nibabel as nib
import SimpleITK as sitk

import torch
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Subset

import monai
from monai.config.deviceconfig import print_config
from monai.utils.misc import set_determinism

from skimage.transform import radon, iradon
from skimage.metrics import (
    peak_signal_noise_ratio as psnr,
    structural_similarity as ssim,
)
from skimage.filters import threshold_otsu
from skimage.morphology import ball
from skimage.segmentation import watershed
from scipy import ndimage

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    mutual_info_classif,
)
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, StratifiedKFold

try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from radiomics import featureextractor

    HAS_PYRAD = True
except ImportError:
    HAS_PYRAD = False
    print("PyRadiomics no disponible. pip install pyradiomics")

from monai.apps.datasets import DecathlonDataset
from monai.data.dataset import CacheDataset
from monai.data.dataloader import DataLoader
from monai.data.utils import decollate_batch
from monai.inferers.inferer import SlidingWindowInferer
from monai.losses.dice import DiceCELoss
from monai.metrics.meandice import DiceMetric
from monai.networks.nets.dynunet import DynUNet
from monai.networks.nets.segresnet import SegResNet
from monai.networks.nets.unet import UNet

from monai.transforms.compose import Compose
from monai.transforms.io.dictionary import LoadImaged
from monai.transforms.spatial.dictionary import (
    Orientationd,
    Spacingd,
    RandFlipd,
    RandRotate90d,
    RandAffined,
)
from monai.transforms.utility.dictionary import (
    EnsureChannelFirstd,
    EnsureTyped,
)
from monai.transforms.intensity.dictionary import (
    ScaleIntensityRanged,
    RandShiftIntensityd,
    RandGaussianNoised,
    RandGaussianSmoothd,
)
from monai.transforms.post.array import (
    AsDiscrete,
    KeepLargestConnectedComponent,
)

from monai.transforms.croppad.dictionary import (
    CropForegroundd,
    RandCropByPosNegLabeld,
)

Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Verificando librerías...
[OK] numpy 2.0.2
[OK] matplotlib
[OK] scipy 1.16.3
[OK] PIL
[OK] sklearn 1.6.1
[OK] pandas 2.3.3
[OK] nibabel 5.3.3
[OK] cv2
[OK] torch 2.10.0+cu128
[OK] torchvision 0.25.0+cu128
[OK] transformers 5.0.0
[OK] huggingface_hub 1.4.1
[OK] accelerate 1.12.0
[OK] sentencepiece 0.2.1
[OK] einops 0.8.2
[OK] open_clip
[OK] gradio 5.50.0
[OK] monai 1.5.2
[OK] SimpleITK 2.5.3
[OK] fire
[OK] radiomics 3.1.1.dev111+g8ed579383


In [45]:
# ==============================================================================
# CONFIGURACIÓN GLOBAL
# ==============================================================================

# --- Rutas ---
RAIZ = Path(".")
DIR_DATOS = RAIZ / "data"
DIR_RESULTADOS = RAIZ / "resultados"
DIR_FIGURAS = DIR_RESULTADOS / "figuras"
DIR_METRICAS = DIR_RESULTADOS / "metricas"
DIR_MODELOS = RAIZ / "modelos"

for d in [DIR_DATOS, DIR_FIGURAS, DIR_METRICAS, DIR_MODELOS]:
    d.mkdir(parents=True, exist_ok=True)

# --- Reproducibilidad ---
SEMILLA = 42
set_determinism(seed=SEMILLA)

# --- Hardware ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_CPU = os.cpu_count() or 1
N_WORKERS = min(4, max(1, N_CPU - 1))

print(f"Dispositivo: {DEVICE}")
if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)} "
        f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)"
    )
print(f"CPUs: {N_CPU}, Workers: {N_WORKERS}")
print_config()

# --- Dataset ---
TASK = "Task06_Lung"
SPACING = (1.5, 1.5, 2.0)  # mm
PATCH_SIZE = (96, 96, 96)
BATCH_SIZE = 2

# --- Ventanas CT clínicas (Window Width, Window Level) ---
VENTANAS_CT = {
    "pulmon": (1500, -600),
    "mediastinica": (400, 40),
    "hueso": (1800, 400),
    "tumor_pulmonar": (600, 40),
}

RANGOS_HU = {
    "aire": (-1050, -950),
    "pulmon": (-900, -400),
    "grasa": (-150, -50),
    "agua": (-20, 20),
    "tejido_blando": (20, 80),
    "tumor_tipico": (-30, 70),
    "hueso_esponjoso": (130, 300),
    "hueso_cortical": (300, 1500),
}

# --- Entrenamiento ---
NUM_EPOCAS = 300
NUM_EPOCAS_ABLACION = 100  # Ajustar a 300 si hay tiempo de GPU
LR = 1e-4
WEIGHT_DECAY = 1e-5
VAL_INTERVAL = 10

# --- Reconstrucción ---
FILTROS_FBP = {
    "Ram-Lak": "ramp",
    "Shepp-Logan": "shepp-logan",
    "Hamming": "hamming",
}
NIVELES_ANGULOS = [180, 90, 45, 30]

# --- Registro ---
N_VOLUMENES_REGISTRO = 5

# --- Ablación ---
FRACCIONES_DATOS = [0.10, 0.25, 0.50, 1.00]
NOMBRES_REGIMEN = {1: "Sin augmentación", 2: "Estándar", 3: "Agresiva"}

# --- Colores ---
COLORES = {
    "DynUNet": "#ef5350",
    "SegResNet": "#4fc3f7",
    "UNet": "#66bb6a",
    "Ram-Lak": "#ef5350",
    "Shepp-Logan": "#4fc3f7",
    "Hamming": "#66bb6a",
}

Dispositivo: cuda
GPU: Tesla T4 (15.6 GB)
CPUs: 4, Workers: 3
MONAI version: 1.5.2
Numpy version: 2.0.2
Pytorch version: 2.10.0+cu128
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: d18565fb3e4fd8c556707f91ac280a2dc3f681c1
MONAI __file__: /usr/local/lib/python3.12/dist-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.5.3
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.3.3
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 11.3.0
Tensorboard version: 2.19.0
gdown version: 5.2.1
TorchVision version: 0.25.0+cu128
tqdm version: 4.67.3
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 5.9.5
pandas version: 2.3.3
einops version: 0.8.2
transformers version: 5.0.0
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visi

In [46]:
# ==============================================================================
# FUNCIONES AUXILIARES
# ==============================================================================

ESTILO_OSCURO = {
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor": "#16213e",
    "axes.edgecolor": "#e0e0e0",
    "axes.labelcolor": "#e0e0e0",
    "text.color": "#e0e0e0",
    "xtick.color": "#e0e0e0",
    "ytick.color": "#e0e0e0",
    "grid.color": "#2a2a4a",
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.titlesize": 16,
    "legend.facecolor": "#16213e",
    "legend.edgecolor": "#4a4a6a",
}


def estilo_oscuro():
    plt.rcParams.update(ESTILO_OSCURO)


def ventanear(imagen, w, l):
    """Aplica ventana CT (Window Width / Window Level) a imagen en HU → [0, 1]."""
    hu_min, hu_max = l - w / 2.0, l + w / 2.0
    return np.clip((imagen - hu_min) / (hu_max - hu_min), 0, 1)


def visor_triplanar(
    volumen,
    titulo="",
    ventana=None,
    mascara=None,
    alpha_mascara=0.35,
    indices=None,
    spacing=None,
    guardar=None,
):
    """Vista triplanar (axial, coronal, sagital) con overlay de máscara."""
    estilo_oscuro()
    if indices is None:
        indices = tuple(s // 2 for s in volumen.shape)
    iz, iy, ix = indices
    cortes = [volumen[iz, :, :], volumen[:, iy, :], volumen[:, :, ix]]
    nombres = [f"Axial (z={iz})", f"Coronal (y={iy})", f"Sagital (x={ix})"]
    if ventana is not None:
        cortes = [ventanear(c, ventana[0], ventana[1]) for c in cortes]
    if spacing:
        sx, sy, sz = spacing
        aspects = [sy / sx, sz / sx, sz / sy]
    else:
        aspects = ["equal"] * 3
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="#1a1a2e")
    if titulo:
        fig.suptitle(
            titulo, fontsize=16, fontweight="bold", color="#e0e0e0", y=1.02
        )
    cmap_mask = ListedColormap(["none", "#ff4444"])
    for ax, corte, nombre, asp in zip(axes, cortes, nombres, aspects):
        ax.imshow(corte, cmap="gray", origin="lower", aspect=asp)
        if mascara is not None:
            if "Axial" in nombre:
                m = mascara[iz, :, :]
            elif "Coronal" in nombre:
                m = mascara[:, iy, :]
            else:
                m = mascara[:, :, ix]
            if np.sum(m) > 0:
                ax.contour(
                    m,
                    levels=[0.5],
                    colors=["#ff4444"],
                    linewidths=1.5,
                    origin="lower",
                )
                ax.imshow(
                    m,
                    cmap=cmap_mask,
                    alpha=alpha_mascara,
                    origin="lower",
                    aspect=asp,
                    vmin=0,
                    vmax=1,
                )
        ax.set_title(nombre, fontsize=12, color="#e0e0e0", pad=8)
        ax.axis("off")
    plt.tight_layout()
    if guardar:
        fig.savefig(
            guardar,
            dpi=150,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
            edgecolor="none",
        )
    return fig


def mosaico_axial(
    volumen,
    mascara=None,
    z_inicio=0,
    z_fin=None,
    n_cortes=15,
    filas=3,
    ventana=(400, 40),
    titulo="",
    guardar=None,
):
    """Mosaico de cortes axiales equiespaciados."""
    estilo_oscuro()
    if z_fin is None:
        z_fin = volumen.shape[0] - 1
    indices = np.linspace(z_inicio, z_fin, n_cortes, dtype=int)
    cols = n_cortes // filas + (1 if n_cortes % filas else 0)
    fig, axes = plt.subplots(
        filas, cols, figsize=(4 * cols, 4 * filas), facecolor="#1a1a2e"
    )
    if titulo:
        fig.suptitle(
            titulo, fontsize=14, fontweight="bold", color="#e0e0e0", y=1.01
        )
    for idx, ax in enumerate(axes.flat):
        if idx < len(indices):
            zi = indices[idx]
            ax.imshow(
                ventanear(volumen[zi], ventana[0], ventana[1]),
                cmap="gray",
                origin="lower",
            )
            if mascara is not None and np.sum(mascara[zi]) > 0:
                ax.contour(
                    mascara[zi],
                    levels=[0.5],
                    colors=["#ff4444"],
                    linewidths=1,
                    origin="lower",
                )
                for sp in ax.spines.values():
                    sp.set_edgecolor("#ff4444")
                    sp.set_linewidth(2)
            ax.set_title(f"z={zi}", fontsize=9, color="#e0e0e0")
        ax.axis("off")
    plt.tight_layout()
    if guardar:
        fig.savefig(
            guardar,
            dpi=150,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
        )
    return fig


def checkerboard(img1, img2, n=8):
    """Tablero de ajedrez para evaluar registro visualmente."""
    h, w = img1.shape[:2]
    th, tw = h // n, w // n
    out = np.zeros_like(img1)
    for i in range(n):
        for j in range(n):
            y1, y2 = i * th, min((i + 1) * th, h)
            x1, x2 = j * tw, min((j + 1) * tw, w)
            out[y1:y2, x1:x2] = (
                img1[y1:y2, x1:x2] if (i + j) % 2 == 0 else img2[y1:y2, x1:x2]
            )
    return out


def estadisticas_tumor(volumen, mascara, spacing):
    """Métricas clínicas de un tumor segmentado."""
    vol_voxel_cm3 = float(np.prod(spacing)) / 1000.0
    tumor_mask = mascara > 0
    n_tumor = int(np.sum(tumor_mask))
    hu_tumor = volumen[tumor_mask]
    n_pulmon = int(np.sum((volumen > -950) & (volumen < -400) & (~tumor_mask)))
    coords = np.argwhere(tumor_mask)
    if len(coords) > 0:
        rango_mm = (coords.max(axis=0) - coords.min(axis=0)) * np.array(
            spacing
        )
        diam = float(np.linalg.norm(rango_mm))
        vol_mm3 = n_tumor * float(np.prod(spacing))
        vol_bbox = float(np.prod(rango_mm + np.array(spacing)))
        esfericidad = vol_mm3 / max(vol_bbox, 1e-8)
    else:
        diam, esfericidad = 0.0, 0.0
    return {
        "volumen_tumor_cm3": round(n_tumor * vol_voxel_cm3, 3),
        "volumen_pulmon_cm3": round(n_pulmon * vol_voxel_cm3, 1),
        "ratio_tumor_pulmon": round(n_tumor / max(n_pulmon, 1), 6),
        "hu_media": round(float(np.mean(hu_tumor)), 2)
        if len(hu_tumor)
        else 0.0,
        "hu_std": round(float(np.std(hu_tumor)), 2) if len(hu_tumor) else 0.0,
        "hu_min": float(np.min(hu_tumor)) if len(hu_tumor) else 0.0,
        "hu_max": float(np.max(hu_tumor)) if len(hu_tumor) else 0.0,
        "hu_mediana": float(np.median(hu_tumor)) if len(hu_tumor) else 0.0,
        "num_voxeles": n_tumor,
        "diametro_max_mm": round(diam, 2),
        "esfericidad_aprox": round(esfericidad, 4),
    }


def dice_score(pred, gt):
    """Dice Similarity Coefficient."""
    pb, gb = pred > 0, gt > 0
    inter = np.sum(pb & gb)
    suma = np.sum(pb) + np.sum(gb)
    if suma == 0:
        return 1.0 if np.sum(gb) == 0 else 0.0
    return 2.0 * inter / suma


def sensibilidad(pred, gt):
    tp = np.sum((pred > 0) & (gt > 0))
    fn = np.sum((pred == 0) & (gt > 0))
    return tp / max(tp + fn, 1)


def especificidad(pred, gt):
    tn = np.sum((pred == 0) & (gt == 0))
    fp = np.sum((pred > 0) & (gt == 0))
    return tn / max(tn + fp, 1)


def hausdorff_95(pred, gt, spacing=(1, 1, 1)):
    """Hausdorff Distance al percentil 95."""
    from scipy.ndimage import distance_transform_edt, binary_erosion

    pb, gb = pred > 0, gt > 0
    if not np.any(pb) or not np.any(gb):
        return float("inf")
    dt_gt = distance_transform_edt(~gb, sampling=spacing)
    dt_pred = distance_transform_edt(~pb, sampling=spacing)
    surf_pred = pb & ~binary_erosion(pb)
    surf_gt = gb & ~binary_erosion(gb)
    if not np.any(surf_pred) or not np.any(surf_gt):
        return float("inf")
    return round(
        float(
            max(
                np.percentile(dt_gt[surf_pred], 95),
                np.percentile(dt_pred[surf_gt], 95),
            )
        ),
        2,
    )


def evaluar_segmentacion(pred, gt, spacing=(1, 1, 1)):
    return {
        "dice": round(dice_score(pred, gt), 4),
        "hd95_mm": hausdorff_95(pred, gt, spacing),
        "sensibilidad": round(sensibilidad(pred, gt), 4),
        "especificidad": round(especificidad(pred, gt), 4),
        "voxeles_pred": int(np.sum(pred > 0)),
        "voxeles_gt": int(np.sum(gt > 0)),
    }


def ncc(a, b):
    """Correlación cruzada normalizada."""
    an = (a - a.mean()) / (a.std() + 1e-8)
    bn = (b - b.mean()) / (b.std() + 1e-8)
    return float(np.mean(an * bn))


def guardar_json(datos, ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, indent=2, ensure_ascii=False, default=str)
    print(f"  → {Path(ruta).name}")


def guardar_csv(df, ruta):
    df.to_csv(ruta, index=True, encoding="utf-8")
    print(f"  → {Path(ruta).name}")


def guardar_figura(fig, ruta, dpi=150):
    fig.savefig(
        ruta,
        dpi=dpi,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
        edgecolor="none",
    )
    plt.close(fig)
    print(f"  → {Path(ruta).name}")


def encontrar_volumen_con_tumor(dataset, max_buscar=20):
    for i in range(min(max_buscar, len(dataset))):
        if np.sum(dataset[i]["label"].numpy()) > 100:
            return i
    return 0


def corte_con_max_tumor(mascara):
    if np.sum(mascara > 0) == 0:
        return mascara.shape[0] // 2
    return int(np.argmax(np.sum(mascara > 0, axis=(1, 2))))


def centroide_tumor(mascara):
    if np.sum(mascara > 0) > 0:
        return tuple(np.argwhere(mascara > 0).mean(axis=0).astype(int))
    return tuple(s // 2 for s in mascara.shape)


def rango_tumor_axial(mascara, margen_pct=0.2):
    if np.sum(mascara > 0) == 0:
        return mascara.shape[0] // 4, 3 * mascara.shape[0] // 4
    zs = np.where(np.sum(mascara > 0, axis=(1, 2)) > 0)[0]
    rango = zs.max() - zs.min()
    margen = max(5, int(margen_pct * rango))
    return max(0, zs.min() - margen), min(
        mascara.shape[0] - 1, zs.max() + margen
    )


def mapear_en_paralelo(elementos, funcion, max_workers):
    """Ejecuta funcion sobre cada elemento en paralelo con ThreadPool."""
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        yield from pool.map(funcion, elementos)

---
# Fase 1: Exploración de datos (P01)

## Headers NIfTI 

Los primero que hacemos al cargar el dataset es inspeccionar los headers de los archivos NIfTI. Cada volumen tiene metadatos imprescindibles: las dimensiones, el *spacing* (tamaño de cada vóxel en mm), la orientación espacial y la matriz afín que mapea coordenadas de vóxel a coordenadas físicas.

Aplicamos dos transforms de MONAI:

- `Orientationd(axcodes="RAS")`: reorienta al espacio estándar Right-Anterior-Superior.
- `Spacingd(pixdim=(1.5, 1.5, 2.0))`: resamplea a spacing fijo.

## Unidades Hounsfield

Las Unidades Hounsfield (HU) son la escala de atenuación de rayos X en CT:

$$\text{HU}=1000\times \dfrac{\mu_{\text{tejido}} - \mu_{\text{agua}}}{\mu_{\text{aire}}}$$

Cada tejido tiene un rango HU: aire $\approx -1000$, grasa $-150$ a $-50$, agua $\approx 0$, tejidos blandos $20-80$, hueso cortical $>300$. Los tumores pulmonares están entre $-30$ y $+70$ HU, solapándose con otros tejidos blandos.

## Ventaneo CT

El ventaneo es la operación más básica en radiología CT: mapea un subconjunto del rango HU a [0, 1] para visualización. Se define por *Window Width* (W, contraste) y *Window Level* (L, brillo). El mismo corte bajo diferentes ventanas revela estructuras completamente distintas.

In [47]:
# ==============================================================================
# FASE 1 — EXPLORACIÓN DE DATOS
# ==============================================================================

transforms_exp = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)

t0 = time.time()
dataset = DecathlonDataset(
    root_dir=str(DIR_DATOS),
    task=TASK,
    section="training",
    transform=transforms_exp,
    download=True,
    cache_rate=0.0,
    num_workers=0,
)
print(f"Dataset cargado: {len(dataset)} volúmenes en {time.time() - t0:.1f}s")

2026-04-06 11:18:21,736 - INFO - Verified 'Task06_Lung.tar', md5: 8afd997733c7fc0432f71255ba4e52dc.
2026-04-06 11:18:21,737 - INFO - File exists: data/Task06_Lung.tar, skipped downloading.
2026-04-06 11:18:21,738 - INFO - Non-empty folder exists in data/Task06_Lung, skipped extracting.
Dataset cargado: 51 volúmenes en 17.4s


In [ ]:
# --- Inspección de headers NIfTI ---
muestra_0 = dataset[0]
meta = muestra_0["image"].meta
ruta_nifti = str(meta.get("filename_or_obj", ""))

info_header = {"monai_version": monai.__version__}
if Path(ruta_nifti).exists():
    nii = nib.load(ruta_nifti)
    h = nii.header
    info_header.update(
        {
            "archivo": Path(ruta_nifti).name,
            "shape_original": list(nii.shape),
            "dtype": str(h.get_data_dtype()),
            "spacing_original_mm": [
                round(float(z), 4) for z in h.get_zooms()[:3]
            ],
            "orientacion_original": list(nib.aff2axcodes(nii.affine)),
            "fov_original_mm": [
                round(float(s * z), 1)
                for s, z in zip(nii.shape[:3], h.get_zooms()[:3])
            ],
        }
    )

img_post = muestra_0["image"].numpy().squeeze()
info_header["shape_post_transform"] = list(img_post.shape)
info_header["spacing_post_transform"] = list(SPACING)
guardar_json(info_header, DIR_METRICAS / "f1_header_ejemplo.json")

print("Header del volumen: 0")
for k, v in info_header.items():
    if k != "affine":
        print(f"  {k}: {v}")

  → f1_header_ejemplo.json
Header del volumen: 0
  monai_version: 1.5.2
  archivo: lung_014.nii.gz
  shape_original: [512, 512, 589]
  dtype: float32
  spacing_original_mm: [0.7422, 0.7422, 0.625]
  orientacion_original: ['L', 'A', 'S']
  fov_original_mm: [380.0, 380.0, 368.1]
  shape_post_transform: [254, 254, 185]
  spacing_post_transform: [1.5, 1.5, 2.0]


In [ ]:
# --- Tabla resumen de todos los volúmenes ---

N_EXPLORAR = len(dataset)
registros = []
print(f"Explorando {N_EXPLORAR} volúmenes...")

for i in range(N_EXPLORAR):
    m = dataset[i]
    img = m["image"].numpy().squeeze()
    msk = m["label"].numpy().squeeze()
    n_tumor = int(np.sum(msk > 0))
    registros.append(
        {
            "volumen": f"lung_{i:03d}",
            "shape_D": img.shape[0],
            "shape_H": img.shape[1],
            "shape_W": img.shape[2],
            "hu_min": round(float(img.min()), 1),
            "hu_max": round(float(img.max()), 1),
            "hu_media": round(float(img.mean()), 1),
            "tiene_tumor": int(n_tumor > 0),
            "voxeles_tumor": n_tumor,
            "volumen_tumor_cm3": round(n_tumor * np.prod(SPACING) / 1000, 2),
        }
    )
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{N_EXPLORAR}")

df_resumen = pd.DataFrame(registros)
guardar_csv(df_resumen, DIR_METRICAS / "f1_resumen_volumenes.csv")
print(
    f"\nVolúmenes con tumor: {int(df_resumen['tiene_tumor'].sum())}/{N_EXPLORAR}"
)
df_resumen.head(10)

Explorando 51 volúmenes...
  10/51
  20/51
  30/51


In [ ]:
# --- Volumen de ejemplo con tumor ---

idx_tumor = encontrar_volumen_con_tumor(dataset)
muestra = dataset[idx_tumor]
vol = muestra["image"].numpy().squeeze()
msk = muestra["label"].numpy().squeeze()
print(
    f"Volumen ejemplo: {idx_tumor}, shape={vol.shape}, tumor={int(np.sum(msk > 0)):,} vóxeles"
)

In [ ]:
# --- Histograma de HU ---

estilo_oscuro()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6), facecolor="#1a1a2e")

hu = vol.flatten()
hu = hu[(hu > -1100) & (hu < 800)]
ax1.hist(
    hu, bins=300, color="#4fc3f7", alpha=0.8, edgecolor="none", density=True
)
bandas = {
    "Aire": (-1050, -950, "#78909c"),
    "Pulmón": (-900, -400, "#66bb6a"),
    "Grasa": (-150, -50, "#ffd54f"),
    "Tejido blando": (20, 80, "#ef5350"),
    "Hueso": (200, 600, "#ab47bc"),
}
for nombre, (lo, hi, color) in bandas.items():
    ax1.axvspan(lo, hi, alpha=0.12, color=color, label=nombre)
ax1.set_xlabel("Unidades Hounsfield (HU)")
ax1.set_ylabel("Densidad")
ax1.set_title("Histograma global de HU")
ax1.legend(fontsize=8)
ax1.set_xlim(-1100, 800)

hu_tumor = vol[msk > 0]
hu_pulm = vol[(vol > -950) & (vol < -400) & (msk == 0)]
if len(hu_tumor) > 10:
    ax2.hist(
        hu_pulm,
        bins=150,
        color="#66bb6a",
        alpha=0.7,
        density=True,
        edgecolor="none",
        label=f"Parénquima (n={len(hu_pulm):,})",
    )
    ax2.hist(
        hu_tumor,
        bins=80,
        color="#ef5350",
        alpha=0.8,
        density=True,
        edgecolor="none",
        label=f"Tumor (n={len(hu_tumor):,})",
    )
    ax2.axvline(
        np.mean(hu_pulm),
        color="#66bb6a",
        ls="--",
        lw=2,
        label=f"μ pulmón: {np.mean(hu_pulm):.0f} HU",
    )
    ax2.axvline(
        np.mean(hu_tumor),
        color="#ef5350",
        ls="--",
        lw=2,
        label=f"μ tumor: {np.mean(hu_tumor):.0f} HU",
    )
    ax2.legend(fontsize=8)
ax2.set_xlabel("HU")
ax2.set_ylabel("Densidad")
ax2.set_title("Tumor vs. parénquima")
plt.tight_layout()
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f1_histograma_hu.png")

In [ ]:
# --- Comparativa de ventanas CT ---

z_tumor = corte_con_max_tumor(msk)
corte_hu = vol[z_tumor]
corte_msk = msk[z_tumor]

estilo_oscuro()
ventanas_demo = [
    ("pulmon", "Pulmón\nW=1500 L=−600"),
    ("mediastinica", "Mediastínica\nW=400 L=40"),
    ("hueso", "Hueso\nW=1800 L=400"),
    ("tumor_pulmonar", "Tumor\nW=600 L=40"),
]
fig, axes = plt.subplots(1, 4, figsize=(24, 6), facecolor="#1a1a2e")
for ax, (vnombre, vtitulo) in zip(axes, ventanas_demo):
    w, l = VENTANAS_CT[vnombre]
    ax.imshow(ventanear(corte_hu, w, l), cmap="gray", origin="lower")
    if np.sum(corte_msk) > 0:
        ax.contour(
            corte_msk,
            levels=[0.5],
            colors=["#ff4444"],
            linewidths=1.5,
            origin="lower",
        )
    ax.set_title(vtitulo, fontsize=11, color="#e0e0e0")
    ax.axis("off")
plt.tight_layout()
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f1_ventanas_ct.png")

In [ ]:
# --- Visor triplanar centrado en el tumor ---

centro = centroide_tumor(msk)
fig = visor_triplanar(
    vol,
    titulo=f"Volumen {idx_tumor} — ventana mediastínica",
    ventana=VENTANAS_CT["mediastinica"],
    mascara=msk,
    indices=centro,
    spacing=SPACING,
    guardar=str(DIR_FIGURAS / "f1_triplanar.png"),
)

In [ ]:
# --- Mosaico de cortes axiales ---

z0, z1 = rango_tumor_axial(msk)
fig = mosaico_axial(
    vol,
    mascara=msk,
    z_inicio=z0,
    z_fin=z1,
    titulo=f"Mosaico axial — Volumen {idx_tumor}",
    guardar=str(DIR_FIGURAS / "f1_mosaico.png"),
)

In [ ]:
# --- Estadísticas volumétricas de todos los tumores ---

stats_lista = []
for i in range(len(dataset)):
    m = dataset[i]
    img_i = m["image"].numpy().squeeze()
    msk_i = m["label"].numpy().squeeze()
    if np.sum(msk_i > 0) > 10:
        s = estadisticas_tumor(img_i, msk_i, SPACING)
        s["volumen_id"] = i
        stats_lista.append(s)

df_stats = pd.DataFrame(stats_lista)
if not df_stats.empty:
    guardar_csv(df_stats, DIR_METRICAS / "f1_estadisticas_tumor.csv")
    print(f"Tumores encontrados: {len(df_stats)}")
    print(
        f"Volumen medio: {df_stats['volumen_tumor_cm3'].mean():.2f} ± "
        f"{df_stats['volumen_tumor_cm3'].std():.2f} cm³"
    )

In [ ]:
# --- Distribuciones de métricas tumorales ---

if not df_stats.empty and len(df_stats) > 2:
    estilo_oscuro()
    fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor="#1a1a2e")
    distribuciones = [
        ("volumen_tumor_cm3", "Volumen (cm³)", "#ef5350"),
        ("diametro_max_mm", "Diámetro máx. (mm)", "#4fc3f7"),
        ("hu_media", "HU media", "#66bb6a"),
        ("hu_std", "HU desv. estándar", "#ffd54f"),
        ("esfericidad_aprox", "Esfericidad", "#ab47bc"),
        ("num_voxeles", "Nº vóxeles", "#ff8a65"),
    ]
    for ax, (col, xlabel, color) in zip(axes.flat, distribuciones):
        ax.set_facecolor("#16213e")
        data = df_stats[col].dropna()
        ax.hist(
            data,
            bins=min(15, len(data)),
            color=color,
            alpha=0.8,
            edgecolor="#16213e",
        )
        ax.axvline(
            data.mean(),
            color="white",
            ls="--",
            lw=1.5,
            label=f"μ={data.mean():.2f}",
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Frecuencia")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show(fig)
    guardar_figura(fig, DIR_FIGURAS / "f1_distribuciones.png")

print("Fase 1 completada.")

---
# Fase 2: Reconstrucción CT simulada (P02)

## Transformada de Radon y FBP

En un escáner CT real, el tubo de rayos X rota alrededor del paciente generando proyecciones desde múltiples ángulos. El conjunto de proyecciones forma el *sinograma*.
La retroproyección filtrada (FBP) invierte la transformada de Radon aplicando un filtro en el dominio de frecuencia:

$$\mathcal{R}f(\theta, s) = \int\!\!\int f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

Comparamos tres filtros:
- **Ram-Lak** ($H(\omega)=|\omega|$): máxima resolución, máximo ruido.
- **Shepp-Logan**: atenuación suave de altas frecuencias.
- **Hamming**: máximo suavizado, menor resolución.

## Submuestreo angular

Reducir el número de proyecciones equivale a reducir la dosis de radiación (CT de baja dosis), pero introduce artefactos de *streak*. Evaluamos 12 combinaciones (3 filtros $\times$ 4 niveles: 180, 90, 45, 30 proyecciones).

In [ ]:
# ==============================================================================
# FASE 2 — RECONSTRUCCIÓN CT
# ==============================================================================

# Reutilizamos vol y msk de la Fase 1
corte_orig = ventanear(vol[z_tumor], 400, 40)


def recortar(rec, orig):
    """
    Recorta o rellena la reconstrucción al tamaño del original para
    evitar errores de shape al calcular el PSNR y SSIM.
    """
    ho, wo = orig.shape
    hr, wr = rec.shape

    # Ajustar altura (eje Y)
    if hr > ho:
        yo = (hr - ho) // 2
        rec = rec[yo : yo + ho, :]
    elif hr < ho:
        pad_y = ho - hr
        pad_top = pad_y // 2
        pad_bottom = pad_y - pad_top
        rec = np.pad(rec, ((pad_top, pad_bottom), (0, 0)), mode="constant")

    # Ajustar anchura (eje X)
    hr, wr = rec.shape
    if wr > wo:
        xo = (wr - wo) // 2
        rec = rec[:, xo : xo + wo]
    elif wr < wo:
        pad_x = wo - wr
        pad_left = pad_x // 2
        pad_right = pad_x - pad_left
        rec = np.pad(rec, ((0, 0), (pad_left, pad_right)), mode="constant")

    return np.clip(rec, 0, 1)


# --- Sinogramas a distintos niveles ---
sinogramas = {}
for n_ang in NIVELES_ANGULOS:
    angulos = np.linspace(0, 180, n_ang, endpoint=False)
    sinogramas[n_ang] = (
        angulos,
        radon(corte_orig, theta=angulos, circle=True),
    )

# --- Demo: original → sinograma → FBP ---
ang_ref, sino_ref = sinogramas[180]
rec_ref = recortar(
    iradon(sino_ref, theta=ang_ref, filter_name="ramp", circle=True),
    corte_orig,
)

estilo_oscuro()
fig, axes = plt.subplots(1, 4, figsize=(24, 6), facecolor="#1a1a2e")

axes[0].imshow(corte_orig, cmap="gray", origin="lower")
if np.sum(corte_msk) > 0:
    axes[0].contour(
        corte_msk,
        levels=[0.5],
        colors=["#ff4444"],
        linewidths=1.5,
        origin="lower",
    )
axes[0].set_title("Corte CT original")
axes[0].axis("off")

axes[1].imshow(
    sino_ref, cmap="hot", aspect="auto", extent=[0, 180, 0, sino_ref.shape[0]]
)
axes[1].set_title("Sinograma (180 proy.)")
axes[1].set_xlabel("Ángulo $\\theta$ (°)")
axes[1].set_ylabel("Detector")

# Retroproyección SIN filtrar (para comparar)
rec_nofilt = recortar(
    iradon(sino_ref, theta=ang_ref, filter_name=None, circle=True), corte_orig
)
axes[2].imshow(rec_nofilt, cmap="gray", origin="lower")
axes[2].set_title("Retroproyección\n(sin filtro)")
axes[2].axis("off")

axes[3].imshow(rec_ref, cmap="gray", origin="lower")
axes[3].set_title(
    f"FBP Ram-Lak\nPSNR={psnr(corte_orig, rec_ref, data_range=1):.1f}dB"
)
axes[3].axis("off")

fig.suptitle(
    "Del corte CT al sinograma y de vuelta (pipeline de reconstrucción)",
    fontsize=15,
    fontweight="bold",
    color="#e0e0e0",
    y=1.02,
)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f2_pipeline.png")

In [ ]:
# --- 12 experimentos: filtros × submuestreos ---

resultados_fbp, reconstrucciones = [], {}
for nombre_f, id_f in FILTROS_FBP.items():
    for n_ang in NIVELES_ANGULOS:
        angulos, sino = sinogramas[n_ang]
        rec = recortar(
            iradon(sino, theta=angulos, filter_name=id_f, circle=True),
            corte_orig,
        )
        val_psnr = psnr(corte_orig, rec, data_range=1.0)
        val_ssim = ssim(corte_orig, rec, data_range=1.0)
        resultados_fbp.append(
            {
                "filtro": nombre_f,
                "n_angulos": n_ang,
                "PSNR_dB": round(val_psnr, 2),
                "SSIM": round(val_ssim, 4),
                "MAE": round(float(np.mean(np.abs(corte_orig - rec))), 5),
            }
        )
        reconstrucciones[(nombre_f, n_ang)] = rec

df_fbp = pd.DataFrame(resultados_fbp)
guardar_csv(df_fbp, DIR_METRICAS / "f2_matriz_fbp.csv")
df_fbp

In [ ]:
# --- Gran matriz visual ---

estilo_oscuro()
nf, nn = len(FILTROS_FBP), len(NIVELES_ANGULOS)
fig, axes = plt.subplots(nf, nn, figsize=(5 * nn, 5 * nf), facecolor="#1a1a2e")
for i, nombre_f in enumerate(FILTROS_FBP):
    for j, n_ang in enumerate(NIVELES_ANGULOS):
        ax = axes[i, j]
        rec = reconstrucciones[(nombre_f, n_ang)]
        row = df_fbp[
            (df_fbp["filtro"] == nombre_f) & (df_fbp["n_angulos"] == n_ang)
        ].iloc[0]
        ax.imshow(rec, cmap="gray", origin="lower")
        s = row["SSIM"]
        color = "#66bb6a" if s > 0.9 else "#ffd54f" if s > 0.7 else "#ef5350"
        ax.set_title(
            f"PSNR={row['PSNR_dB']}dB | SSIM={s}", fontsize=9, color=color
        )
        ax.axis("off")
        if j == 0:
            ax.set_ylabel(
                nombre_f,
                fontsize=13,
                color="#e0e0e0",
                rotation=90,
                labelpad=15,
            )
        if i == 0:
            ax.text(
                0.5,
                1.12,
                f"{n_ang} proy.",
                transform=ax.transAxes,
                ha="center",
                fontsize=11,
                color="#e0e0e0",
            )
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f2_matriz_filtros.png", dpi=120)

In [ ]:
# --- Curvas de calidad ---

estilo_oscuro()
fig, axes = plt.subplots(1, 3, figsize=(22, 6), facecolor="#1a1a2e")
for nombre_f in FILTROS_FBP:
    color = COLORES[nombre_f]
    sub = df_fbp[df_fbp["filtro"] == nombre_f].sort_values("n_angulos")
    axes[0].plot(
        sub["n_angulos"],
        sub["PSNR_dB"],
        marker="o",
        lw=2.5,
        ms=8,
        color=color,
        label=nombre_f,
    )
    axes[1].plot(
        sub["n_angulos"],
        sub["SSIM"],
        marker="s",
        lw=2.5,
        ms=8,
        color=color,
        label=nombre_f,
    )
    axes[2].plot(
        sub["n_angulos"],
        sub["MAE"],
        marker="D",
        lw=2.5,
        ms=8,
        color=color,
        label=nombre_f,
    )
for ax, ylabel, title in zip(
    axes,
    ["PSNR (dB)", "SSIM", "MAE"],
    [
        "Peak Signal-to-Noise Ratio",
        "Structural Similarity",
        "Mean Absolute Error",
    ],
):
    ax.set_facecolor("#16213e")
    ax.set_xlabel("Nº proyecciones")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xticks(NIVELES_ANGULOS)
axes[1].axhline(0.7, color="#ffd54f", ls="--", alpha=0.5)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f2_curvas_calidad.png")

In [ ]:
# --- Artefactos de streak ---

estilo_oscuro()
fig, axes = plt.subplots(2, 3, figsize=(20, 13), facecolor="#1a1a2e")
ref = reconstrucciones[("Ram-Lak", 180)]
for j, n_ang in enumerate([90, 45, 30]):
    rec = reconstrucciones[("Ram-Lak", n_ang)]
    diff = rec - ref
    row = df_fbp[
        (df_fbp["filtro"] == "Ram-Lak") & (df_fbp["n_angulos"] == n_ang)
    ].iloc[0]
    axes[0, j].imshow(rec, cmap="gray", origin="lower")
    axes[0, j].set_title(f"{n_ang} proy. | PSNR={row['PSNR_dB']}dB")
    axes[0, j].axis("off")
    im = axes[1, j].imshow(
        diff, cmap="RdBu_r", origin="lower", vmin=-0.25, vmax=0.25
    )
    axes[1, j].set_title(f"Artefactos (MAE={np.mean(np.abs(diff)):.4f})")
    axes[1, j].axis("off")
fig.colorbar(im, ax=axes[1, :], fraction=0.015, pad=0.02, label="Diferencia")
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f2_artefactos.png")

print("Fase 2 completada.")

---
# Fase 3: Registro de imágenes médicas (P03)

## Registro rígido y afín

El registro busca la transformación $T^*$ que mejor alinea una imagen *móvil* con una *fija*: $T^* = \arg\min_T \mathcal{D}(F, M \circ T)$. Implementamos dos pasos:

1. **Rígido (6 GDL):** 3 rotaciones + 3 traslaciones. Corrige posicionamiento.
2. **Afín (12 GDL):** + 3 escalados + 3 cizallas. Corrige tamaño corporal.

Ambos usan información mutua de Mattes, gradient descent y pirámide multi-resolución
(shrink 4→2→1). También registramos las máscaras tumorales para medir el Dice pre/post.

## Atlas poblacional

Promediando los volúmenes registrados obtenemos la anatomía "promedio" del dataset, junto con la variabilidad inter-paciente (desviación estándar vóxel a vóxel).

In [ ]:
# ==============================================================================
# FASE 3 — REGISTRO
# ==============================================================================

sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(max(1, N_CPU // 2))
IDX_REF = 0
vols_np_r, msks_np_r, vols_sitk_r, msks_sitk_r = {}, {}, {}, {}

for i in range(N_VOLUMENES_REGISTRO + 1):
    m = dataset[i]
    v = m["image"].numpy().squeeze()
    mk = m["label"].numpy().squeeze()
    vols_np_r[i] = v
    msks_np_r[i] = mk
    vs = sitk.GetImageFromArray(v.astype(np.float32))
    vs.SetSpacing(SPACING)
    vs.SetOrigin((0, 0, 0))
    vols_sitk_r[i] = vs
    ms = sitk.GetImageFromArray(mk.astype(np.float32))
    ms.CopyInformation(vs)
    msks_sitk_r[i] = ms
    print(
        f"  Vol {i}: shape={v.shape}, tumor={'Sí' if np.sum(mk > 0) > 0 else 'No'}"
    )

In [ ]:
def registrar_rigido(fija, movil):
    """
    Registro rígido 3D con información mutua de Mattes.
    6 GDL: 3 rotaciones (Euler) + 3 traslaciones.
    Pirámide multi-resolución 4→2→1.
    Devuelve (imagen_registrada, transformación, métricas_convergencia).
    """
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.1)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsGradientDescent(
        learningRate=1.0,
        numberOfIterations=200,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=10,
    )
    reg.SetOptimizerScalesFromPhysicalShift()

    tf = sitk.Euler3DTransform()
    reg.SetInitialTransform(
        sitk.CenteredTransformInitializer(
            fija, movil, tf, sitk.CenteredTransformInitializerFilter.GEOMETRY
        )
    )
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([4, 2, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    convergencia = []
    reg.AddCommand(
        sitk.sitkIterationEvent,
        lambda: convergencia.append(reg.GetMetricValue()),
    )

    t0 = time.time()
    tf_final = reg.Execute(fija, movil)
    dt = time.time() - t0

    img = sitk.Resample(
        movil, fija, tf_final, sitk.sitkBSpline, 0.0, movil.GetPixelID()
    )
    return (
        img,
        tf_final,
        {
            "tiempo_s": dt,
            "convergencia": convergencia,
            "metrica": reg.GetMetricValue(),
            "iteraciones": reg.GetOptimizerIteration(),
        },
    )


def registrar_afin(fija, movil, tf_rigida=None):
    """
    Registro afín 3D: 12 GDL (rígido + 3 escalados + 3 cizallas).
    Inicializado desde el resultado rígido si se proporciona.
    """
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.1)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsGradientDescent(
        learningRate=0.5,
        numberOfIterations=200,
        convergenceMinimumValue=1e-6,
        convergenceWindowSize=10,
    )
    reg.SetOptimizerScalesFromPhysicalShift()

    tf_afin = sitk.AffineTransform(3)
    if tf_rigida is not None:
        comp = sitk.CompositeTransform(3)
        comp.AddTransform(tf_rigida)
        comp.AddTransform(tf_afin)
        reg.SetInitialTransform(comp, inPlace=False)
    else:
        reg.SetInitialTransform(
            sitk.CenteredTransformInitializer(
                fija,
                movil,
                tf_afin,
                sitk.CenteredTransformInitializerFilter.GEOMETRY,
            )
        )

    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([4, 2, 0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    convergencia = []
    reg.AddCommand(
        sitk.sitkIterationEvent,
        lambda: convergencia.append(reg.GetMetricValue()),
    )

    t0 = time.time()
    tf_final = reg.Execute(fija, movil)
    dt = time.time() - t0

    img = sitk.Resample(
        movil, fija, tf_final, sitk.sitkBSpline, 0.0, movil.GetPixelID()
    )
    return (
        img,
        tf_final,
        {
            "tiempo_s": dt,
            "convergencia": convergencia,
            "metrica": reg.GetMetricValue(),
        },
    )


In [ ]:
def registrar_caso(i):
    img_rig, tf_rig, met_rig = registrar_rigido(fija, vols_sitk_r[i])
    img_afin, tf_afin, met_afin = registrar_afin(fija, vols_sitk_r[i], tf_rig)

    # Imagen y máscara antes del registro en el espacio de referencia (identidad)
    ident_tf = sitk.Transform()
    img_antes = sitk.Resample(
        vols_sitk_r[i],
        fija,
        ident_tf,
        sitk.sitkLinear,
        0.0,
        vols_sitk_r[i].GetPixelID(),
    )
    msk_antes = sitk.Resample(
        msks_sitk_r[i],
        fija,
        ident_tf,
        sitk.sitkNearestNeighbor,
        0.0,
        msks_sitk_r[i].GetPixelID(),
    )

    # Registrar máscara con nearest neighbor
    msk_reg = sitk.Resample(
        msks_sitk_r[i],
        fija,
        tf_afin,
        sitk.sitkNearestNeighbor,
        0.0,
        msks_sitk_r[i].GetPixelID(),
    )

    antes_np = sitk.GetArrayFromImage(img_antes)
    rig_np = sitk.GetArrayFromImage(img_rig)
    afin_np = sitk.GetArrayFromImage(img_afin)
    msk_antes_np = sitk.GetArrayFromImage(msk_antes)
    msk_reg_np = sitk.GetArrayFromImage(msk_reg)

    resultado = {
        "antes_np": antes_np,
        "rigido_np": rig_np,
        "afin_np": afin_np,
        "msk_reg_np": msk_reg_np,
        "conv_rig": met_rig["convergencia"],
        "conv_afin": met_afin["convergencia"],
    }

    # Métricas
    ncc_antes = ncc(fija_np, antes_np)
    ncc_rig = ncc(fija_np, rig_np)
    ncc_afin = ncc(fija_np, afin_np)

    # Dice de máscaras tumorales
    dice_antes = dice_score(msk_antes_np, msk_fija)
    dice_post = dice_score(msk_reg_np, msk_fija)

    metrica = {
        "volumen": i,
        "NCC_antes": round(ncc_antes, 4),
        "NCC_rigido": round(ncc_rig, 4),
        "NCC_afin": round(ncc_afin, 4),
        "mejora_NCC_pct": round(
            (ncc_afin - ncc_antes) / max(abs(ncc_antes), 1e-8) * 100, 1
        ),
        "Dice_tumor_antes": round(dice_antes, 4),
        "Dice_tumor_post": round(dice_post, 4),
        "tiempo_rigido_s": round(met_rig["tiempo_s"], 1),
        "tiempo_afin_s": round(met_afin["tiempo_s"], 1),
        "iter_rigido": met_rig["iteraciones"],
    }
    return i, resultado, metrica

In [ ]:
# --- Ejecutar registro para todos los volúmenes ---

fija = vols_sitk_r[IDX_REF]
fija_np = sitk.GetArrayFromImage(fija)
msk_fija = msks_np_r[IDX_REF]
resultados = {}
metricas = []

indices_moviles = [i for i in range(N_VOLUMENES_REGISTRO + 1) if i != IDX_REF]

for i, resultado, metrica in mapear_en_paralelo(
    indices_moviles, registrar_caso, N_WORKERS
):
    resultados[i] = resultado
    metricas.append(metrica)
    print(f"\nRegistrando volumen {i} ⟶ referencia {IDX_REF}...")
    print(
        f"  NCC: {metrica['NCC_antes']:.4f} → {metrica['NCC_afin']:.4f} | "
        f"Dice tumor: {metrica['Dice_tumor_antes']:.4f} → "
        f"{metrica['Dice_tumor_post']:.4f}"
    )

df_reg = pd.DataFrame(metricas)
guardar_csv(df_reg, DIR_METRICAS / "f3_metricas_registro.csv")

# Alias de compatibilidad con nombres usados en celdas anteriores.
resultados_reg = resultados
fija_np_r = fija_np

In [ ]:
# --- Visualización: checkerboard ---

resultados_vis = globals().get("resultados", globals().get("resultados_reg"))
fija_np_vis = globals().get("fija_np", globals().get("fija_np_r"))
if resultados_vis is None or fija_np_vis is None:
    raise RuntimeError("Ejecuta antes la celda de registro de la Fase 3.")

idx_ej = list(resultados_vis.keys())[0]
z_mid = fija_np_vis.shape[0] // 2
c_fija = ventanear(fija_np_vis[z_mid], 400, 40)
c_movil = ventanear(resultados_vis[idx_ej]["antes_np"][z_mid], 400, 40)
c_rig = ventanear(resultados_vis[idx_ej]["rigido_np"][z_mid], 400, 40)
c_afin = ventanear(resultados_vis[idx_ej]["afin_np"][z_mid], 400, 40)

estilo_oscuro()
fig, axes = plt.subplots(2, 4, figsize=(24, 12), facecolor="#1a1a2e")
for ax, img, tit in zip(
    axes[0],
    [c_fija, c_movil, c_rig, c_afin],
    ["Fija", "Móvil en ref.", "Rígido (6 GDL)", "Afín (12 GDL)"],
):
    ax.imshow(img, cmap="gray", origin="lower")
    ax.set_title(tit)
    ax.axis("off")
for ax, img, tit in zip(
    axes[1],
    [
        np.abs(c_fija - c_movil),
        checkerboard(c_fija, c_movil),
        checkerboard(c_fija, c_rig),
        checkerboard(c_fija, c_afin),
    ],
    ["Diferencia", "CB antes", "CB rígido", "CB afín"],
):
    cmap = "hot" if "Dif" in tit else "gray"
    ax.imshow(img, cmap=cmap, origin="lower")
    ax.set_title(tit)
    ax.axis("off")
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f3_registro.png")

In [ ]:
# --- Atlas poblacional ---

resultados_vis = globals().get("resultados", globals().get("resultados_reg"))
fija_np_vis = globals().get("fija_np", globals().get("fija_np_r"))
if resultados_vis is None or fija_np_vis is None:
    raise RuntimeError("Ejecuta antes la celda de registro de la Fase 3.")

atlas = fija_np_vis.astype(np.float64).copy()
n_contrib = 1
for res in resultados_vis.values():
    atlas += res["afin_np"].astype(np.float64)
    n_contrib += 1
atlas = (atlas / n_contrib).astype(np.float32)
stack = np.stack(
    [fija_np_vis] + [r["afin_np"] for r in resultados_vis.values()]
)
atlas_std = np.std(stack, axis=0)

estilo_oscuro()
fig, axes = plt.subplots(2, 3, figsize=(18, 12), facecolor="#1a1a2e")
zz, yy, xx = [s // 2 for s in atlas.shape]
for j, (nombre, cm, cs) in enumerate(
    [
        ("Axial", atlas[zz], atlas_std[zz]),
        ("Coronal", atlas[:, yy, :], atlas_std[:, yy, :]),
        ("Sagital", atlas[:, :, xx], atlas_std[:, :, xx]),
    ]
):
    axes[0, j].imshow(ventanear(cm, 400, 40), cmap="gray", origin="lower")
    axes[0, j].set_title(f"{nombre} — Media")
    axes[0, j].axis("off")
    im = axes[1, j].imshow(cs, cmap="magma", origin="lower")
    axes[1, j].set_title(f"{nombre} — Variabilidad")
    axes[1, j].axis("off")
    plt.colorbar(im, ax=axes[1, j], fraction=0.046)
fig.suptitle(
    f"Atlas poblacional ({n_contrib} volúmenes)",
    fontsize=15,
    fontweight="bold",
    color="#e0e0e0",
    y=1.01,
)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f3_atlas.png")

print("Fase 3 completada.")

---
# Fase 4: Segmentación: clásica vs. deep learning (P04)

Esta es la fase más importante del proyecto. Comparamos tres baselines clásicos (Otsu, crecimiento de regiones, watershed) con tres arquitecturas de deep learning (DynUNet, SegResNet, UNet) entrenadas con MONAI.

## Pipeline de preprocesamiento

Un punto que nos costó depurar: al principio usábamos `RandSpatialCropd`, que recorta parches en posiciones aleatorias. Como los tumores son muy pequeños (<0.1% del volumen), la mayoría de parches no contenían tumor y la red aprendía a predecir todo como fondo. La solución fue `RandCropByPosNegLabeld` con `pos=1, neg=1`: el 50% de parches se centran en tumor, el 50% en fondo. Este cambio fue el que más impacto tuvo.

## Arquitecturas

- **DynUNet**: tipo nnU-Net con deep supervision.
- **SegResNet**: bloques residuales, eficiente en VRAM.
- **UNet**: baseline clásica de 5 niveles.

Configuración: DiceCELoss, AdamW (lr=1e-4), CosineAnnealing, AMP, batch=2, parches 96³.

In [ ]:
# ==============================================================================
# FASE 4 — SEGMENTACIÓN
# ==============================================================================

# --- Baselines clásicos ---
def segmentar_otsu(vol_hu):
    mask = (vol_hu > -30) & (vol_hu < 100) & (vol_hu > -400)
    mask = ndimage.binary_opening(mask, ball(2))
    mask = ndimage.binary_closing(mask, ball(2))
    labels, n = ndimage.label(mask)
    if n == 0:
        return np.zeros_like(mask, dtype=np.float32)
    sizes = np.bincount(labels.ravel())
    keep = sizes > 100
    keep[0] = False
    return keep[labels].astype(np.float32)


def segmentar_watershed(vol_hu):
    v = np.clip(vol_hu, -100, 200)
    v = (v - v.min()) / (v.max() - v.min() + 1e-8)
    grad = ndimage.sobel(v)
    suave = ndimage.gaussian_filter(v, sigma=2.0)
    thr = threshold_otsu(suave[suave > 0.1]) if np.any(suave > 0.1) else 0.5
    markers = ndimage.label(suave > thr)[0]
    ws = watershed(grad, markers=markers, compactness=0.01)
    out = np.zeros_like(vol_hu, dtype=np.float32)
    for lid in np.unique(ws):
        if lid == 0:
            continue
        region = ws == lid
        if -30 < np.mean(vol_hu[region]) < 100 and 50 < np.sum(region) < 50000:
            out[region] = 1.0
    return out

In [ ]:
# --- Transforms ---

train_tf = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=-1024,
            a_max=600,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        RandCropByPosNegLabeld(
            keys=["image", "label"],
            label_key="label",
            spatial_size=PATCH_SIZE,
            pos=1,
            neg=1,
            num_samples=2,
            image_key="image",
            image_threshold=0,
        ),
        RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.2),
        RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.2),
        RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.2),
        RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
        RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        EnsureTyped(keys=["image", "label"]),
    ]
)

val_tf = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),
        ScaleIntensityRanged(
            keys=["image"],
            a_min=-1024,
            a_max=600,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        EnsureTyped(keys=["image", "label"]),
    ]
)

In [ ]:
# --- Dataloaders ---

ds_train = DecathlonDataset(
    root_dir=str(DIR_DATOS),
    task=TASK,
    section="training",
    transform=train_tf,
    download=False,
    cache_rate=1.0,
    num_workers=2,
)
ds_val = DecathlonDataset(
    root_dir=str(DIR_DATOS),
    task=TASK,
    section="training",
    transform=val_tf,
    download=False,
    cache_rate=1.0,
    num_workers=2,
)

n_total = len(ds_train)
n_val = max(1, int(n_total * 0.2))
n_train = n_total - n_val
train_loader = DataLoader(
    Subset(ds_train, range(n_train)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    Subset(ds_val, range(n_train, n_total)),
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
print(f"Train: {n_train} | Val: {n_val}")

In [ ]:
# --- Arquitecturas ---


def crear_dynunet():
    ks = [[3, 3, 3]] * 5
    ss = [[1, 1, 1], [2, 2, 2], [2, 2, 2], [2, 2, 2], [2, 2, 2]]
    return DynUNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        kernel_size=ks,
        strides=ss,
        upsample_kernel_size=ss[1:],
        norm_name="instance",
        deep_supervision=True,
        deep_supr_num=3,
    )


def crear_segresnet():
    return SegResNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        init_filters=16,
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
        dropout_prob=0.2,
    )


def crear_unet():
    return UNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        channels=(16, 32, 64, 128, 256),
        strides=(2, 2, 2, 2),
        num_res_units=2,
        norm="instance",
    )


for nombre, fn in [
    ("DynUNet", crear_dynunet),
    ("SegResNet", crear_segresnet),
    ("UNet", crear_unet),
]:
    print(f"  {nombre}: {sum(p.numel() for p in fn().parameters()):,} params")

In [ ]:
# --- Entrenamiento ---


def entrenar(modelo, nombre, deep_sup=False, num_epocas=NUM_EPOCAS):
    modelo = modelo.to(DEVICE)
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
    optim = torch.optim.AdamW(
        modelo.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=num_epocas, eta_min=1e-7
    )
    scaler = GradScaler()
    dice_m = DiceMetric(include_background=False, reduction="mean")
    inferer = SlidingWindowInferer(
        roi_size=PATCH_SIZE, sw_batch_size=4, overlap=0.5
    )
    post_p = Compose([AsDiscrete(argmax=True, to_onehot=2)])
    post_l = Compose([AsDiscrete(to_onehot=2)])
    mejor_dice, mejor_ep = 0.0, 0
    hist = {"train_loss": [], "val_dice": [], "lr": []}
    for ep in range(num_epocas):
        modelo.train()
        loss_ep, nb = 0.0, 0
        for batch in train_loader:
            imgs, lbls = batch["image"].to(DEVICE), batch["label"].to(DEVICE)
            optim.zero_grad()
            with autocast():
                preds = modelo(imgs)
                if deep_sup:
                    if isinstance(preds, (list, tuple)):
                        loss = sum(loss_fn(p, lbls) for p in preds) / len(preds)
                    elif preds.dim() == lbls.dim() + 1:
                        # DynUNet apila las cabezas en dim 1: [B, heads, C, H, W, D]
                        heads = torch.unbind(preds, dim=1)
                        loss = sum(loss_fn(h, lbls) for h in heads) / len(heads)
                    else:
                        loss = loss_fn(preds, lbls)
                else:
                    loss = loss_fn(preds, lbls)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            loss_ep += loss.item()
            nb += 1
        sched.step()
        hist["train_loss"].append(loss_ep / max(nb, 1))
        hist["lr"].append(sched.get_last_lr()[0])
        dice_val = 0.0
        if (ep + 1) % VAL_INTERVAL == 0 or ep == num_epocas - 1:
            modelo.eval()
            dice_m.reset()
            with torch.no_grad():
                for bv in val_loader:
                    iv, lv = bv["image"].to(DEVICE), bv["label"].to(DEVICE)
                    pv = inferer(iv, modelo)
                    if isinstance(pv, (list, tuple)):
                        pv = pv[0]
                    dice_m(
                        [post_p(p) for p in decollate_batch(pv)],
                        [post_l(l) for l in decollate_batch(lv)],
                    )
            dice_val = dice_m.aggregate().item()
            if dice_val > mejor_dice:
                mejor_dice, mejor_ep = dice_val, ep + 1
                torch.save(
                    modelo.state_dict(),
                    str(DIR_MODELOS / f"mejor_{nombre}.pth"),
                )
            print(
                f"  {nombre} ep {ep + 1:>3d}/{num_epocas} | loss {hist['train_loss'][-1]:.4f} "
                f"| dice {dice_val:.4f} | mejor {mejor_dice:.4f}"
            )
        hist["val_dice"].append(dice_val)
    hist["mejor_dice"] = mejor_dice
    hist["mejor_epoca"] = mejor_ep
    hist["n_params"] = sum(p.numel() for p in modelo.parameters())
    return hist

In [ ]:
# --- Entrenar las tres arquitecturas ---

historiales = {}
historiales["DynUNet"] = entrenar(crear_dynunet(), "DynUNet", deep_sup=True)
historiales["SegResNet"] = entrenar(crear_segresnet(), "SegResNet")
historiales["UNet"] = entrenar(crear_unet(), "UNet")

guardar_json(
    {
        k: {
            kk: vv
            for kk, vv in v.items()
            if kk not in ("train_loss", "val_dice", "lr")
        }
        for k, v in historiales.items()
    },
    DIR_METRICAS / "f4_historiales.json",
)

In [ ]:
# --- Evaluación baselines clásicos sobre validación ---

val_tf_raw = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)
ds_val_raw = DecathlonDataset(
    root_dir=str(DIR_DATOS),
    task=TASK,
    section="training",
    transform=val_tf_raw,
    download=False,
    cache_rate=0.0,
    num_workers=0,
)

eval_clasicos = {"Otsu": [], "Watershed": []}
for i in range(n_train, min(n_total, n_train + n_val)):
    m = ds_val_raw[i]
    v = m["image"].numpy().squeeze()
    gt = m["label"].numpy().squeeze()
    if np.sum(gt > 0) < 10:
        continue
    eval_clasicos["Otsu"].append(
        evaluar_segmentacion(segmentar_otsu(v), gt, SPACING)
    )
    eval_clasicos["Watershed"].append(
        evaluar_segmentacion(segmentar_watershed(v), gt, SPACING)
    )

filas_comp = []
for nombre, evals in eval_clasicos.items():
    if evals:
        df_e = pd.DataFrame(evals)
        filas_comp.append(
            {
                "metodo": nombre,
                "tipo": "Clásico",
                "dice_medio": round(df_e["dice"].mean(), 4),
                "sensibilidad": round(df_e["sensibilidad"].mean(), 4),
                "parametros": 0,
            }
        )
for nombre, h in historiales.items():
    filas_comp.append(
        {
            "metodo": nombre,
            "tipo": "Deep Learning",
            "dice_medio": h["mejor_dice"],
            "sensibilidad": 0,
            "parametros": h["n_params"],
        }
    )

df_comp = pd.DataFrame(filas_comp)
guardar_csv(df_comp, DIR_METRICAS / "f4_comparativa.csv")

In [ ]:
# --- Curvas de entrenamiento ---

estilo_oscuro()
fig, axes = plt.subplots(1, 3, figsize=(22, 6), facecolor="#1a1a2e")
for nombre, h in historiales.items():
    c = COLORES[nombre]
    axes[0].plot(h["train_loss"], color=c, label=nombre, lw=2, alpha=0.8)
    dvals = [(i + 1, d) for i, d in enumerate(h["val_dice"]) if d > 0]
    if dvals:
        axes[1].plot(
            *zip(*dvals), color=c, label=nombre, lw=2, marker="o", ms=4
        )
    axes[2].plot(h["lr"], color=c, label=nombre, lw=2, alpha=0.8)
for ax, yl, tit in zip(
    axes,
    ["DiceCE Loss", "Dice", "Learning Rate"],
    ["Pérdida", "Dice validación", "LR schedule"],
):
    ax.set_facecolor("#16213e")
    ax.set_xlabel("Época")
    ax.set_ylabel(yl)
    ax.set_title(tit)
    ax.legend()
    ax.grid(alpha=0.3)
axes[1].set_ylim(0, 1)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f4_curvas.png")

In [ ]:
# --- Barplot ---
estilo_oscuro()
fig, ax = plt.subplots(figsize=(12, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
metodos = df_comp["metodo"].tolist()
dices = df_comp["dice_medio"].tolist()
colores_bar = [COLORES.get(m, "#999999") for m in metodos]
bars = ax.bar(metodos, dices, color=colores_bar, alpha=0.85)
for bar, d in zip(bars, dices):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{d:.4f}",
        ha="center",
        fontsize=11,
        color="#e0e0e0",
        fontweight="bold",
    )
ax.set_ylabel("Dice")
ax.set_title("Comparativa de segmentación")
ax.set_ylim(0, 1)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f4_barplot.png")

print("Fase 4 completada.")

---
# Fase 5: Análisis radiómico y fusión con DL (P04)

## ¿Qué es la radiómica?

La radiómica extrae features cuantitativos de imágenes médicas que capturan propiedades del tumor invisibles al ojo humano. PyRadiomics implementa siete categorías IBSI (~1.200 features con filtros wavelet y LoG). Los seleccionamos con ANOVA, información mutua y LASSO, y los combinamos con deep features del bottleneck de SegResNet para clasificación con Random Forest y XGBoost.

In [ ]:
# ==============================================================================
# FASE 5 — RADIÓMICA Y FUSIÓN
# ==============================================================================

sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(max(1, N_CPU // 2))


def extraer_radiomics(vol_np, msk_np, spacing=SPACING):
    if not HAS_PYRAD or np.sum(msk_np > 0) < 10:
        return {}
    vs = sitk.GetImageFromArray(vol_np.astype(np.float32))
    vs.SetSpacing(spacing)
    ms = sitk.GetImageFromArray((msk_np > 0).astype(np.uint8))
    ms.CopyInformation(vs)
    ext = featureextractor.RadiomicsFeatureExtractor(
        binWidth=25, normalize=True, normalizeScale=100, removeOutliers=3
    )
    ext.enableAllFeatures()
    ext.enableImageTypeByName("Wavelet")
    ext.enableImageTypeByName("LoG", customArgs={"sigma": [1.0, 2.0, 3.0]})
    try:
        res = ext.execute(vs, ms)
    except Exception:
        return {}
    return {
        k: float(v)
        for k, v in res.items()
        if not k.startswith("diagnostics")
        and isinstance(v, (int, float, np.floating))
    }

In [ ]:
N_MAX = min(30, len(dataset))
feats_list, ids_rad = [], []
if HAS_PYRAD:
    for i in range(N_MAX):
        m = dataset[i]
        v, mk = m["image"].numpy().squeeze(), m["label"].numpy().squeeze()
        if np.sum(mk > 0) > 50:
            f = extraer_radiomics(v, mk)
            if f:
                feats_list.append(f)
                ids_rad.append(i)
                print(f"  Vol {i}: {len(f)} features")
    if feats_list:
        df_rad = pd.DataFrame(feats_list, index=ids_rad)
        guardar_csv(df_rad, DIR_METRICAS / "f5_features_radiomicos.csv")
        print(f"Radiómicos: {df_rad.shape}")
    else:
        df_rad = pd.DataFrame()
else:
    df_rad = pd.DataFrame()

In [ ]:
# --- Selección de features ---

seleccion_info = {}
if not df_rad.empty and df_rad.shape[0] > 5:
    df_clean = df_rad.dropna(axis=1).loc[:, lambda x: x.std() > 1e-8]
    shape_cols = [
        c for c in df_clean.columns if "shape" in c.lower() and "Volume" in c
    ]
    if shape_cols:
        y_rad = (
            (df_clean[shape_cols[0]] > df_clean[shape_cols[0]].median())
            .astype(int)
            .values
        )
    else:
        y_rad = np.random.randint(0, 2, len(df_clean))
    X_rad_all = StandardScaler().fit_transform(df_clean.values)
    # ANOVA
    sel = SelectKBest(f_classif, k=min(20, X_rad_all.shape[1]))
    sel.fit(X_rad_all, y_rad)
    seleccion_info["top20_anova"] = {
        k: round(v, 2)
        for k, v in pd.Series(sel.scores_, index=df_clean.columns)
        .nlargest(20)
        .items()
    }
    # MI
    mi = mutual_info_classif(X_rad_all, y_rad, random_state=42)
    seleccion_info["top20_mi"] = {
        k: round(v, 4)
        for k, v in pd.Series(mi, index=df_clean.columns).nlargest(20).items()
    }
    # LASSO
    try:
        lasso = LassoCV(cv=3, random_state=42, max_iter=5000).fit(
            X_rad_all, y_rad
        )
        seleccion_info["lasso_nonzero"] = int(np.sum(lasso.coef_ != 0))
    except:
        pass
    guardar_json(seleccion_info, DIR_METRICAS / "f5_seleccion.json")

In [ ]:
# --- Deep features del bottleneck ---

modelo_path = DIR_MODELOS / "mejor_SegResNet.pth"
deep_feats = []
if modelo_path.exists() and ids_rad:
    modelo_sr = SegResNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        init_filters=16,
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
    ).to(DEVICE)
    modelo_sr.load_state_dict(
        torch.load(str(modelo_path), map_location=DEVICE)
    )
    modelo_sr.eval()
    activaciones = {}
    hook = modelo_sr.down_layers[-1].register_forward_hook(
        lambda mod, inp, out: activaciones.update({"act": out.detach().cpu()})
    )
    with torch.no_grad():
        for idx in ids_rad:
            m = dataset[idx]
            img = m["image"].unsqueeze(0).to(DEVICE)
            if any(s > 200 for s in img.shape[2:]):
                img = img[:, :, :96, :96, :96]
            try:
                _ = modelo_sr(img)
                deep_feats.append(
                    torch.mean(activaciones["act"], dim=(2, 3, 4))
                    .squeeze()
                    .numpy()
                )
            except:
                deep_feats.append(np.zeros(128))
    hook.remove()
    if deep_feats:
        df_deep = pd.DataFrame(
            deep_feats,
            columns=[f"deep_{j}" for j in range(len(deep_feats[0]))],
            index=ids_rad[: len(deep_feats)],
        )
        guardar_csv(df_deep, DIR_METRICAS / "f5_deep_features.csv")

In [ ]:
# --- Fusión y clasificación ---

clf_resultados = {}
if not df_rad.empty and df_rad.shape[0] > 5 and deep_feats:
    common = list(set(ids_rad[: len(deep_feats)]) & set(df_clean.index))
    if len(common) > 5:
        X_r = StandardScaler().fit_transform(
            df_clean.loc[common].values[:, :20]
        )
        X_d = StandardScaler().fit_transform(
            np.array(deep_feats[: len(common)])
        )
        X_f = np.hstack([X_r, X_d])
        y_s = y_rad[: len(common)]
        cv = StratifiedKFold(
            n_splits=min(3, len(y_s) // 2), shuffle=True, random_state=42
        )
        for etiq, Xd in [("radiomica", X_r), ("deep", X_d), ("fusion", X_f)]:
            for cn, co in [
                ("RF", RandomForestClassifier(100, random_state=42))
            ] + (
                [("XGB", XGBClassifier(100, random_state=42, verbosity=0))]
                if HAS_XGB
                else []
            ):
                try:
                    sc = cross_val_score(
                        co, Xd, y_s, cv=cv, scoring="accuracy"
                    )
                    clf_resultados[f"{etiq}_{cn}"] = {
                        "accuracy": round(sc.mean(), 4),
                        "std": round(sc.std(), 4),
                    }
                except:
                    pass
        guardar_json(clf_resultados, DIR_METRICAS / "f5_clasificacion.json")
        print("Clasificación:", clf_resultados)

In [ ]:
# --- Visualización: PCA + t-SNE ---

if not df_rad.empty and df_rad.shape[0] > 5:
    estilo_oscuro()
    fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor="#1a1a2e")
    pca_out = PCA(n_components=2, random_state=42).fit_transform(X_rad_all)
    axes[0].set_facecolor("#16213e")
    axes[0].scatter(
        pca_out[:, 0],
        pca_out[:, 1],
        c=y_rad,
        cmap="RdYlGn",
        s=100,
        edgecolors="#e0e0e0",
        lw=0.5,
    )
    axes[0].set_title("PCA")
    perp = min(5, max(2, len(df_clean) // 3))
    tsne_out = TSNE(
        n_components=2, random_state=42, perplexity=perp
    ).fit_transform(X_rad_all)
    axes[1].set_facecolor("#16213e")
    axes[1].scatter(
        tsne_out[:, 0],
        tsne_out[:, 1],
        c=y_rad,
        cmap="RdYlGn",
        s=100,
        edgecolors="#e0e0e0",
        lw=0.5,
    )
    axes[1].set_title("t-SNE")
    plt.show(fig)
    guardar_figura(fig, DIR_FIGURAS / "f5_tsne_pca.png")

    # Heatmap de correlación
    if "top20_anova" in seleccion_info:
        top_cols = [
            c
            for c in list(seleccion_info["top20_anova"].keys())[:15]
            if c in df_clean.columns
        ]
        if len(top_cols) > 3:
            corr = df_clean[top_cols].corr()
            fig, ax = plt.subplots(figsize=(12, 10), facecolor="#1a1a2e")
            ax.set_facecolor("#16213e")
            im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
            ax.set_xticks(range(len(top_cols)))
            ax.set_yticks(range(len(top_cols)))
            short = [c.split("_")[-1][:20] for c in top_cols]
            ax.set_xticklabels(short, rotation=45, ha="right", fontsize=8)
            ax.set_yticklabels(short, fontsize=8)
            ax.set_title("Correlación entre top features")
            plt.colorbar(im, ax=ax, fraction=0.046)
            plt.show(fig)
            plt.show(fig)
            guardar_figura(fig, DIR_FIGURAS / "f5_correlacion.png")

print("Fase 5 completada.")

---
# Fase 6: Experimento de escasez de datos (P04)

## Diseño experimental

La pregunta central: **¿cuánto ayuda la data augmentation de MONAI cuando hay pocos datos?** Entrenamos SegResNet bajo 12 configuraciones: 4 fracciones (10%, 25%, 50%, 100%) × 3 regímenes de augmentación:

1. **Sin augmentación:** solo preprocesamiento determinista.
2. **Estándar:** flips, rotaciones 90°, shift de intensidad.
3. **Agresiva:** + deformaciones afines, ruido gaussiano, suavizado.

**Nota sobre el debugging:** Esta fase fue la que más problemas nos dio. Además del bug de `RandSpatialCropd` vs `RandCropByPosNegLabeld`, descubrimos que estábamos reutilizando las transforms de entrenamiento (con crop aleatorio) para validación. El `SlidingWindowInferer` recibía un parche de 96³ ya recortado en vez del volumen completo, y el Dice daba 0 siempre. La solución fue crear un pipeline de validación separado sin `RandCropByPosNegLabeld`.

In [ ]:
# ==============================================================================
# FASE 6 — ESCASEZ DE DATOS (12 EXPERIMENTOS)
# ==============================================================================

torch.backends.cudnn.benchmark = True

_base_f6 = [
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(
        keys=["image", "label"], pixdim=SPACING, mode=("bilinear", "nearest")
    ),
    ScaleIntensityRanged(
        keys=["image"], a_min=-1024, a_max=600, b_min=0, b_max=1, clip=True
    ),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),
]

# Transforms de VALIDACIÓN: SIN RandCropByPosNegLabeld.
# El SlidingWindowInferer necesita el volumen completo.
_val_tf_f6 = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(
            keys=["image", "label"],
            pixdim=SPACING,
            mode=("bilinear", "nearest"),
        ),
        ScaleIntensityRanged(
            keys=["image"], a_min=-1024, a_max=600, b_min=0, b_max=1, clip=True
        ),
        CropForegroundd(keys=["image", "label"], source_key="image"),
        EnsureTyped(keys=["image", "label"]),
    ]
)


def make_train_transforms_f6(reg):
    tfs = list(_base_f6)
    if reg >= 2:
        tfs += [
            RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.2),
            RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.2),
            RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.2),
            RandRotate90d(keys=["image", "label"], prob=0.2, max_k=3),
            RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
        ]
    if reg >= 3:
        tfs += [
            RandAffined(
                keys=["image", "label"],
                prob=0.3,
                rotate_range=(0.05, 0.05, 0.05),
                scale_range=(0.1, 0.1, 0.1),
                translate_range=(5, 5, 5),
                mode=("bilinear", "nearest"),
            ),
            RandGaussianNoised(keys=["image"], prob=0.3, mean=0, std=0.05),
            RandGaussianSmoothd(
                keys=["image"],
                prob=0.2,
                sigma_x=(0.5, 1),
                sigma_y=(0.5, 1),
                sigma_z=(0.5, 1),
            ),
        ]
    tfs.append(EnsureTyped(keys=["image", "label"]))
    return Compose(tfs)


In [ ]:
def run_exp_f6(fraccion, regimen):
    """Entrena SegResNet con fracción de datos y régimen de augmentación."""
    ds_tr = DecathlonDataset(
        root_dir=str(DIR_DATOS),
        task=TASK,
        section="training",
        transform=make_train_transforms_f6(regimen),
        download=False,
        cache_rate=0.5,
        num_workers=0,
    )
    # Validación con transforms SEPARADOS (sin crop aleatorio)
    ds_vl = DecathlonDataset(
        root_dir=str(DIR_DATOS),
        task=TASK,
        section="training",
        transform=_val_tf_f6,
        download=False,
        cache_rate=0.5,
        num_workers=0,
    )

    n = len(ds_tr)
    n_v = max(1, int(n * 0.2))
    n_tf = n - n_v
    n_sub = max(1, int(n_tf * fraccion))
    rng = np.random.RandomState(42)
    idx_tr = rng.permutation(n_tf)[:n_sub].tolist()
    idx_vl = list(range(n_tf, n))

    loader_tr = DataLoader(
        Subset(ds_tr, idx_tr),
        batch_size=2,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    loader_vl = DataLoader(
        Subset(ds_vl, idx_vl),
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    modelo = SegResNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=2,
        init_filters=16,
        blocks_down=(1, 2, 2, 4),
        blocks_up=(1, 1, 1),
        dropout_prob=0.2,
    ).to(DEVICE)
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
    optim = torch.optim.AdamW(
        modelo.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optim, T_max=NUM_EPOCAS_ABLACION
    )
    scaler = GradScaler()
    dice_m = DiceMetric(include_background=False, reduction="mean")
    inferer = SlidingWindowInferer(
        roi_size=PATCH_SIZE, sw_batch_size=4, overlap=0.5
    )
    post_p = Compose([AsDiscrete(argmax=True, to_onehot=2)])
    post_l = Compose([AsDiscrete(to_onehot=2)])

    mejor = 0.0
    t0 = time.time()
    for ep in range(NUM_EPOCAS_ABLACION):
        modelo.train()
        loss_ep, nb = 0, 0
        for b in loader_tr:
            i, l = b["image"].to(DEVICE), b["label"].to(DEVICE)
            optim.zero_grad()
            with autocast():
                loss = loss_fn(modelo(i), l)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            loss_ep += loss.item()
            nb += 1
        sched.step()
        if (ep + 1) % VAL_INTERVAL == 0 or ep == NUM_EPOCAS_ABLACION - 1:
            modelo.eval()
            dice_m.reset()
            with torch.no_grad():
                for bv in loader_vl:
                    pv = inferer(bv["image"].to(DEVICE), modelo)
                    dice_m(
                        [post_p(p) for p in decollate_batch(pv)],
                        [
                            post_l(l)
                            for l in decollate_batch(bv["label"].to(DEVICE))
                        ],
                    )
            dv = dice_m.aggregate().item()
            mejor = max(mejor, dv)
    dt = time.time() - t0
    loss_final = loss_ep / max(nb, 1)
    print(
        f"  frac={fraccion:.0%} reg={regimen} n={n_sub} → Dice={mejor:.4f} ({dt:.0f}s)"
    )
    return {
        "fraccion": fraccion,
        "regimen": regimen,
        "nombre_regimen": NOMBRES_REGIMEN[regimen],
        "n_train": n_sub,
        "mejor_dice": round(mejor, 4),
        "loss_final": round(loss_final, 4),
        "tiempo_s": round(dt, 1),
    }


In [ ]:
resultados_f6 = []
for frac in FRACCIONES_DATOS:
    for reg in [1, 2, 3]:
        resultados_f6.append(run_exp_f6(frac, reg))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

df_abl = pd.DataFrame(resultados_f6)
guardar_csv(df_abl, DIR_METRICAS / "f6_ablacion.csv")
df_abl


In [ ]:
# --- Curvas de aprendizaje ---

estilo_oscuro()
fig, ax = plt.subplots(figsize=(12, 8), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
colores_reg = {1: "#ef5350", 2: "#4fc3f7", 3: "#66bb6a"}
marcadores = {1: "o", 2: "s", 3: "D"}
for reg in [1, 2, 3]:
    sub = df_abl[df_abl["regimen"] == reg].sort_values("fraccion")
    ax.plot(
        sub["fraccion"] * 100,
        sub["mejor_dice"],
        color=colores_reg[reg],
        marker=marcadores[reg],
        lw=2.5,
        ms=10,
        label=NOMBRES_REGIMEN[reg],
    )
    for _, row in sub.iterrows():
        ax.annotate(
            f"{row['mejor_dice']:.3f}",
            xy=(row["fraccion"] * 100, row["mejor_dice"]),
            xytext=(5, 10),
            textcoords="offset points",
            fontsize=9,
            color=colores_reg[reg],
            fontweight="bold",
        )
ax.set_xlabel("Fracción de datos (%)")
ax.set_ylabel("Mejor Dice")
ax.set_title("Curvas de aprendizaje: impacto de la augmentación MONAI")
ax.legend(fontsize=12, loc="lower right")
ax.grid(alpha=0.3)
ax.set_xticks([10, 25, 50, 100])
ax.set_xlim(5, 105)
ax.set_ylim(0, 1)
ax.axhline(0.5, color="#ffd54f", ls="--", alpha=0.5)
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f6_curvas_aprendizaje.png", dpi=200)


In [ ]:
# --- Heatmap ---

estilo_oscuro()
pivot = df_abl.pivot_table(
    values="mejor_dice",
    index="fraccion",
    columns="nombre_regimen",
    aggfunc="first",
)
fig, ax = plt.subplots(figsize=(10, 6), facecolor="#1a1a2e")
ax.set_facecolor("#16213e")
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{int(f * 100)}%" for f in pivot.index])
ax.set_xlabel("Régimen")
ax.set_ylabel("Fracción")
ax.set_title("Heatmap: Dice por configuración")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(
            j,
            i,
            f"{val:.3f}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            color="black" if val > 0.5 else "white",
        )
plt.colorbar(im, ax=ax, fraction=0.046, label="Dice")
plt.show(fig)
guardar_figura(fig, DIR_FIGURAS / "f6_heatmap.png")

print("Fase 6 completada.")


---
# Fase 7: Demo interactiva (P04)

La aplicación Gradio permite probar el pipeline en directo durante la presentación oral. Se sube un NIfTI, se elige ventana y método, y se obtiene la visualización triplanar con overlay y métricas volumétricas. Ejecutar `demo.launch(share=True)` para lanzar.

In [ ]:
# ==============================================================================
# FASE 7 — DEMO GRADIO (OPCIONAL)
# ==============================================================================

try:
    import gradio as gr

    HAS_GRADIO = True
except ImportError:
    HAS_GRADIO = False
    print("Gradio no disponible. pip install gradio")

if HAS_GRADIO:
    from monai.transforms import (
        EnsureChannelFirst,
        Orientation,
        Spacing,
        ScaleIntensityRange,
        EnsureType,
    )

    modelos_demo = {}
    inferer_demo = SlidingWindowInferer(
        roi_size=PATCH_SIZE, sw_batch_size=4, overlap=0.5
    )
    preproc_demo = Compose(
        [
            EnsureChannelFirst(),
            Orientation(axcodes="RAS"),
            Spacing(pixdim=SPACING, mode="bilinear"),
            ScaleIntensityRange(
                a_min=-1024, a_max=600, b_min=0, b_max=1, clip=True
            ),
            EnsureType(),
        ]
    )

    for nombre, fn in [
        (
            "SegResNet",
            lambda: SegResNet(
                spatial_dims=3,
                in_channels=1,
                out_channels=2,
                init_filters=16,
                blocks_down=(1, 2, 2, 4),
                blocks_up=(1, 1, 1),
            ),
        ),
        (
            "DynUNet",
            lambda: DynUNet(
                spatial_dims=3,
                in_channels=1,
                out_channels=2,
                kernel_size=[[3, 3, 3]] * 5,
                strides=[
                    [1, 1, 1],
                    [2, 2, 2],
                    [2, 2, 2],
                    [2, 2, 2],
                    [2, 2, 2],
                ],
                upsample_kernel_size=[[2, 2, 2]] * 4,
                norm_name="instance",
                deep_supervision=False,
            ),
        ),
    ]:
        ruta = DIR_MODELOS / f"mejor_{nombre}.pth"
        if ruta.exists():
            m = fn().to(DEVICE)
            m.load_state_dict(torch.load(str(ruta), map_location=DEVICE))
            m.eval()
            modelos_demo[nombre] = m
            print(f"  {nombre} cargado")

    def seg_otsu_demo(v):
        mask = (v > -30) & (v < 100) & (v > -400)
        mask = ndimage.binary_opening(mask, ball(2))
        mask = ndimage.binary_closing(mask, ball(2))
        labels, n = ndimage.label(mask)
        if n == 0:
            return np.zeros_like(mask, dtype=np.float32)
        sizes = np.bincount(labels.ravel())
        keep = sizes > 100
        keep[0] = False
        return keep[labels].astype(np.float32)

    def pipeline_demo(archivo, ventana_nombre, metodo, z_idx):
        if archivo is None:
            return None, "Suba un archivo NIfTI."
        try:
            vol_hu = nib.load(archivo).get_fdata().astype(np.float32)
            vol_prep = preproc_demo(torch.from_numpy(vol_hu)).numpy().squeeze()
            if "Otsu" in metodo:
                mask = seg_otsu_demo(vol_hu)
            elif "SegResNet" in metodo:
                t = (
                    torch.from_numpy(vol_prep)
                    .unsqueeze(0)
                    .unsqueeze(0)
                    .float()
                    .to(DEVICE)
                )
                with torch.no_grad():
                    pred = inferer_demo(t, modelos_demo["SegResNet"])
                mask = (
                    torch.argmax(pred, dim=1)
                    .squeeze()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )
            else:
                mask = np.zeros_like(vol_hu)
            if mask.shape != vol_hu.shape:
                from scipy.ndimage import zoom as ndzoom

                mask = ndzoom(
                    mask,
                    np.array(vol_hu.shape) / np.array(mask.shape),
                    order=0,
                )
            w, l = VENTANAS_CT.get(ventana_nombre, (400, 40))
            z = min(max(0, z_idx), vol_hu.shape[0] - 1)
            fig, axes = plt.subplots(
                1, 3, figsize=(18, 6), facecolor="#1a1a2e"
            )
            for ax, (nom, corte, m) in zip(
                axes,
                [
                    ("Axial", vol_hu[z], mask[z]),
                    (
                        "Coronal",
                        vol_hu[:, vol_hu.shape[1] // 2, :],
                        mask[:, mask.shape[1] // 2, :],
                    ),
                    (
                        "Sagital",
                        vol_hu[:, :, vol_hu.shape[2] // 2],
                        mask[:, :, mask.shape[2] // 2],
                    ),
                ],
            ):
                ax.set_facecolor("#16213e")
                ax.imshow(ventanear(corte, w, l), cmap="gray", origin="lower")
                if np.sum(m) > 0:
                    ax.contour(
                        m,
                        levels=[0.5],
                        colors=["#ff4444"],
                        linewidths=2,
                        origin="lower",
                    )
                ax.set_title(nom, color="#e0e0e0")
                ax.axis("off")
            plt.tight_layout()
            stats = estadisticas_tumor(vol_hu, mask, SPACING)
            texto = (
                f"**Método:** {metodo}\n\n"
                f"**Vóxeles:** {stats['num_voxeles']:,}\n\n"
                f"**Volumen:** {stats['volumen_tumor_cm3']:.2f} cm³\n\n"
                f"**Diámetro:** {stats['diametro_max_mm']:.1f} mm"
            )
            return fig, texto
        except Exception as e:
            return None, f"Error: {e}"

    metodos_disp = ["Otsu (clásico)"]
    if "SegResNet" in modelos_demo:
        metodos_disp.append("SegResNet (DL)")
    if "DynUNet" in modelos_demo:
        metodos_disp.append("DynUNet (DL)")

    with gr.Blocks(title="CT Pulmonar — MONAI") as demo:
        gr.Markdown("# Pipeline de análisis de tumores pulmonares en CT")
        with gr.Row():
            with gr.Column(scale=1):
                f_in = gr.File(
                    label="Volumen NIfTI", file_types=[".nii", ".nii.gz"]
                )
                v_in = gr.Radio(
                    ["pulmon", "mediastinica", "hueso"],
                    value="mediastinica",
                    label="Ventana",
                )
                m_in = gr.Radio(
                    metodos_disp, value=metodos_disp[0], label="Método"
                )
                z_in = gr.Slider(0, 300, 50, step=1, label="Corte axial (z)")
                btn = gr.Button("Analizar", variant="primary")
            with gr.Column(scale=2):
                fig_out = gr.Plot()
                txt_out = gr.Markdown()
        btn.click(pipeline_demo, [f_in, v_in, m_in, z_in], [fig_out, txt_out])
    print("Demo lista. Ejecutar: demo.launch(share=True)")


---
# Conclusiones

Hemos desarrollado un pipeline que conecta las cuatro prácticas de la asignatura a través del análisis de tumores pulmonares en CT. MONAI ha sido el framework central para carga de datos, preprocesamiento, entrenamiento, inferencia y post-procesamiento.

Las contribuciones que consideramos más valiosas son:

1. La comparativa sistemática clásico/DL con el mismo protocolo de evaluación.
2. El estudio de ablación de 12 experimentos (fracciones × augmentación).
3. La fusión radiómica-DL como prueba de concepto.
4. El proceso de debugging documentado (RandSpatialCropd → RandCropByPosNegLabeld, y la separación de transforms de validación), que nos enseñó que en segmentación médica 3D la coherencia entre pipelines de train/val/inference es crítica.